# Telco Customer Churn - Data Preprocessing

This notebook handles data cleaning, preprocessing, and feature engineering based on insights from EDA.

Note: This notebook covers the next steps mentioned in the EDA section.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pickle
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

print('Libraries imported successfully!')

Libraries imported successfully!


## 1. Load the Dataset

In [2]:
# Load data
df = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
print(f'Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns')
df.head()

Dataset loaded: 7043 rows, 21 columns


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 2. Data Cleaning

In [3]:
# Check initial data types
print('Initial data types:')
print(df.dtypes)

Initial data types:
customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object


In [4]:
# Convert TotalCharges to numeric
print('Converting TotalCharges to numeric...')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Check for missing values after conversion
missing_total_charges = df['TotalCharges'].isna().sum()
print(f'Missing values in TotalCharges: {missing_total_charges}')

if missing_total_charges > 0:
    print(f'Filling {missing_total_charges} missing values with median...')
    df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)
    print('✓ Missing values filled')

Converting TotalCharges to numeric...
Missing values in TotalCharges: 11
Filling 11 missing values with median...
✓ Missing values filled


In [5]:
# Drop customerID (not useful for prediction)
print('Dropping customerID column...')
df = df.drop('customerID', axis=1)
print(f'✓ New shape: {df.shape}')

Dropping customerID column...
✓ New shape: (7043, 20)


In [6]:
# Check for any remaining missing values
print('Missing values check:')
missing = df.isnull().sum()
if missing.sum() == 0:
    print('✓ No missing values!')
else:
    print(missing[missing > 0])

Missing values check:
✓ No missing values!


## 3. Feature Engineering

In [7]:
# Create a copy for processing
df_processed = df.copy()

print('Original features:', df_processed.shape[1])

Original features: 20


In [8]:
# Optional: Create tenure groups
df_processed['tenure_group'] = pd.cut(
    df_processed['tenure'], 
    bins=[0, 12, 24, 48, 72], 
    labels=['0-1 year', '1-2 years', '2-4 years', '4+ years']
)

print('Tenure groups created:')
print(df_processed['tenure_group'].value_counts())

Tenure groups created:
tenure_group
4+ years     2239
0-1 year     2175
2-4 years    1594
1-2 years    1024
Name: count, dtype: int64


In [9]:
# Optional: Create average charges per month
df_processed['avg_charges_per_month'] = df_processed['TotalCharges'] / (df_processed['tenure'] + 1)

print('Average charges per month created')
print(df_processed['avg_charges_per_month'].describe())

Average charges per month created
count    7043.000000
mean       61.173413
std        61.019723
min         9.183333
25%        26.274411
50%        61.150000
75%        84.940047
max      1397.475000
Name: avg_charges_per_month, dtype: float64


## 4. Encode Categorical Variables

In [10]:
# Separate target variable
print('Encoding target variable (Churn)...')
df_processed['Churn'] = df_processed['Churn'].map({'Yes': 1, 'No': 0})
print(f'✓ Churn encoded: {df_processed["Churn"].value_counts().to_dict()}')

Encoding target variable (Churn)...
✓ Churn encoded: {0: 5174, 1: 1869}


In [11]:
# Encode binary categorical variables
print('\nEncoding binary categorical variables...')
binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']

for col in binary_cols:
    if col in df_processed.columns:
        df_processed[col] = df_processed[col].map({'Yes': 1, 'No': 0})
        print(f'✓ {col} encoded')


Encoding binary categorical variables...
✓ Partner encoded
✓ Dependents encoded
✓ PhoneService encoded
✓ PaperlessBilling encoded


In [12]:
# Handle service-related columns with 'No internet service' or 'No phone service'
print('\nHandling service-related columns...')
service_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
                'TechSupport', 'StreamingTV', 'StreamingMovies']

for col in service_cols:
    if col in df_processed.columns:
        df_processed[col] = df_processed[col].replace({'No internet service': 'No'})
        df_processed[col] = df_processed[col].map({'Yes': 1, 'No': 0})
        print(f'✓ {col} encoded')

# Handle MultipleLines
if 'MultipleLines' in df_processed.columns:
    df_processed['MultipleLines'] = df_processed['MultipleLines'].replace({'No phone service': 'No'})
    df_processed['MultipleLines'] = df_processed['MultipleLines'].map({'Yes': 1, 'No': 0})
    print(f'✓ MultipleLines encoded')


Handling service-related columns...
✓ OnlineSecurity encoded
✓ OnlineBackup encoded
✓ DeviceProtection encoded
✓ TechSupport encoded
✓ StreamingTV encoded
✓ StreamingMovies encoded
✓ MultipleLines encoded


In [13]:
# Encode gender
print('\nEncoding gender...')
df_processed['gender'] = df_processed['gender'].map({'Male': 1, 'Female': 0})
print('✓ gender encoded')


Encoding gender...
✓ gender encoded


In [14]:
# One-hot encode multi-category variables
print('\nOne-hot encoding multi-category variables...')
categorical_cols = ['InternetService', 'Contract', 'PaymentMethod']

# Add tenure_group if created
if 'tenure_group' in df_processed.columns:
    categorical_cols.append('tenure_group')

df_processed = pd.get_dummies(df_processed, columns=categorical_cols, drop_first=True)
print(f'✓ Categorical variables encoded')
print(f'Total features after encoding: {df_processed.shape[1]}')


One-hot encoding multi-category variables...
✓ Categorical variables encoded
Total features after encoding: 28


In [15]:
# Display the processed dataframe
print('\nProcessed DataFrame:')
print(df_processed.head())
print(f'\nShape: {df_processed.shape}')
print(f'\nColumns: {df_processed.columns.tolist()}')


Processed DataFrame:
   gender  SeniorCitizen  Partner  Dependents  tenure  PhoneService  \
0       0              0        1           0       1             0   
1       1              0        0           0      34             1   
2       1              0        0           0       2             1   
3       1              0        0           0      45             0   
4       0              0        0           0       2             1   

   MultipleLines  OnlineSecurity  OnlineBackup  DeviceProtection  ...  \
0              0               0             1                 0  ...   
1              0               1             0                 1  ...   
2              0               1             1                 0  ...   
3              0               1             0                 1  ...   
4              0               0             0                 0  ...   

   InternetService_Fiber optic  InternetService_No  Contract_One year  \
0                        False         

## 5. Split Data into Features and Target

In [16]:
# Separate features and target
X = df_processed.drop('Churn', axis=1)
y = df_processed['Churn']

print(f'Features (X): {X.shape}')
print(f'Target (y): {y.shape}')
print(f'\nClass distribution in target:')
print(y.value_counts())
print(f'\nChurn rate: {y.mean():.2%}')

Features (X): (7043, 27)
Target (y): (7043,)

Class distribution in target:
Churn
0    5174
1    1869
Name: count, dtype: int64

Churn rate: 26.54%


## 6. Train-Test Split

In [17]:
# Split data with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print('Data split completed:')
print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set: {X_test.shape[0]} samples')
print(f'\nChurn rate in training set: {y_train.mean():.2%}')
print(f'Churn rate in test set: {y_test.mean():.2%}')

Data split completed:
Training set: 5634 samples
Test set: 1409 samples

Churn rate in training set: 26.54%
Churn rate in test set: 26.54%


## 7. Feature Scaling

In [18]:
# Initialize scaler
scaler = StandardScaler()

# Fit on training data and transform both sets
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame to preserve column names
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

print('✓ Features scaled using StandardScaler')
print(f'\nScaled training set shape: {X_train_scaled.shape}')
print(f'Scaled test set shape: {X_test_scaled.shape}')

✓ Features scaled using StandardScaler

Scaled training set shape: (5634, 27)
Scaled test set shape: (1409, 27)


In [19]:
# Verify scaling
print('Scaled data statistics (training set):')
print(X_train_scaled.describe())

Scaled data statistics (training set):
             gender  SeniorCitizen       Partner    Dependents        tenure  \
count  5.634000e+03   5.634000e+03  5.634000e+03  5.634000e+03  5.634000e+03   
mean   1.135052e-16   7.440898e-17 -7.031018e-17  2.900689e-17 -1.008935e-17   
std    1.000089e+00   1.000089e+00  1.000089e+00  1.000089e+00  1.000089e+00   
min   -1.005696e+00  -4.417730e-01 -9.692341e-01 -6.515565e-01 -1.322329e+00   
25%   -1.005696e+00  -4.417730e-01 -9.692341e-01 -6.515565e-01 -9.559779e-01   
50%    9.943362e-01  -4.417730e-01 -9.692341e-01 -6.515565e-01 -1.418632e-01   
75%    9.943362e-01  -4.417730e-01  1.031742e+00  1.534786e+00  9.164859e-01   
max    9.943362e-01   2.263606e+00  1.031742e+00  1.534786e+00  1.608483e+00   

       PhoneService  MultipleLines  OnlineSecurity  OnlineBackup  \
count  5.634000e+03   5.634000e+03    5.634000e+03  5.634000e+03   
mean  -2.679985e-18  -2.522338e-18    2.143988e-17  2.774572e-17   
std    1.000089e+00   1.000089e+00  

## 8. Save Processed Data

In [20]:
# Save processed data
print('Saving processed data...')

# Save train/test splits
X_train_scaled.to_csv('../data/X_train_scaled.csv', index=False)
X_test_scaled.to_csv('../data/X_test_scaled.csv', index=False)
y_train.to_csv('../data/y_train.csv', index=False)
y_test.to_csv('../data/y_test.csv', index=False)

print('✓ Scaled data saved to data/ folder')

# Save scaler for future use
with open('../models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print('✓ Scaler saved to models/scaler.pkl')

# Save feature names
feature_names = X_train.columns.tolist()
with open('../models/feature_names.pkl', 'wb') as f:
    pickle.dump(feature_names, f)
print('✓ Feature names saved to models/feature_names.pkl')

Saving processed data...
✓ Scaled data saved to data/ folder
✓ Scaler saved to models/scaler.pkl
✓ Feature names saved to models/feature_names.pkl


## 9. Summary

In [21]:
print('\n' + '='*60)
print('DATA PREPROCESSING SUMMARY')
print('='*60)
print(f'\n1. Total samples: {len(df)}')
print(f'2. Total features after preprocessing: {X.shape[1]}')
print(f'3. Training samples: {X_train.shape[0]}')
print(f'4. Test samples: {X_test.shape[0]}')
print(f'5. Churn rate: {y.mean():.2%}')
print(f'6. Missing values: {df_processed.isnull().sum().sum()}')
print(f'\n7. Feature list:')
for i, col in enumerate(X.columns, 1):
    print(f'   {i}. {col}')
print('\n' + '='*60)
print('✓ Data is ready for model training!')
print('='*60)


DATA PREPROCESSING SUMMARY

1. Total samples: 7043
2. Total features after preprocessing: 27
3. Training samples: 5634
4. Test samples: 1409
5. Churn rate: 26.54%
6. Missing values: 0

7. Feature list:
   1. gender
   2. SeniorCitizen
   3. Partner
   4. Dependents
   5. tenure
   6. PhoneService
   7. MultipleLines
   8. OnlineSecurity
   9. OnlineBackup
   10. DeviceProtection
   11. TechSupport
   12. StreamingTV
   13. StreamingMovies
   14. PaperlessBilling
   15. MonthlyCharges
   16. TotalCharges
   17. avg_charges_per_month
   18. InternetService_Fiber optic
   19. InternetService_No
   20. Contract_One year
   21. Contract_Two year
   22. PaymentMethod_Credit card (automatic)
   23. PaymentMethod_Electronic check
   24. PaymentMethod_Mailed check
   25. tenure_group_1-2 years
   26. tenure_group_2-4 years
   27. tenure_group_4+ years

✓ Data is ready for model training!


## Next Steps

The data is now ready for model training. You can:
1. Train XGBoost model
2. Experiment with other algorithms
3. Perform hyperparameter tuning
4. Evaluate model performance
5. Create predictions on new data